# Use Case 2: Fare Prediction Modeling

## Objective

The objective of this notebook is to develop machine learning models for predicting taxi fare amounts before ride initiation.

Based on insights obtained during exploratory data analysis, relevant features are engineered and multiple regression models are trained and evaluated to identify the most effective approach for fare estimation.

# 1. Import Required Libraries

Import libraries required for:

- Data manipulation
- Feature engineering
- Data splitting
- Preprocessing
- Model development
- Model evaluation
- Model persistence

In [3]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")

# 2. Load Dataset

Load the cleaned dataset prepared during preprocessing.

In [4]:
df = pd.read_parquet("/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/USECASE2_Trip_Fare_Forecasting/data/cleaned_taxi_data.parquet")

print(df.shape)

df.head()

(3499962, 10)


,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,fare_amount,cbd_congestion_fee,trip_duration
0,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,239,238,7.2,0.00,5.550000
1,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,142,209,38.7,0.75,42.800000
2,2026-01-01 00:47:11,2026-01-01 01:00:47,2.0,2.33,1.0,144,137,14.2,0.75,13.600000
3,2026-01-01 00:17:54,2026-01-01 00:28:32,1.0,1.30,1.0,142,50,11.4,0.75,10.633333
4,2026-01-01 00:34:14,2026-01-01 01:11:58,1.0,5.34,1.0,161,45,37.3,0.75,37.733333


# 3. Feature Engineering

Based on EDA findings, temporal and geographical features are engineered to capture patterns associated with taxi fares.

Features engineered include:

- Pickup Hour
- Pickup Day of Week
- Pickup Month
- Weekend Indicator
- Rush Hour Indicator
- Night Trip Indicator
- Pickup-Dropoff Location Pair

In [5]:
fare_df = df.copy()

fare_df["tpep_pickup_datetime"] = pd.to_datetime(
    fare_df["tpep_pickup_datetime"]
)

fare_df["pickup_hour"] = (
    fare_df["tpep_pickup_datetime"]
    .dt.hour
)

fare_df["pickup_dayofweek"] = (
    fare_df["tpep_pickup_datetime"]
    .dt.dayofweek
)

fare_df["pickup_month"] = (
    fare_df["tpep_pickup_datetime"]
    .dt.month
)

# Weekend Feature

In [6]:
fare_df["is_weekend"] = (
    fare_df["pickup_dayofweek"]
    .isin([5, 6])
    .astype(int)
)

# Rush Hour Feature

In [7]:
fare_df["is_rush_hour"] = (
    fare_df["pickup_hour"]
    .isin([7, 8, 9, 16, 17, 18, 19])
    .astype(int)
)

# Night Feature

In [8]:
fare_df["is_night"] = (
    (
        fare_df["pickup_hour"] >= 22
    )
    |
    (
        fare_df["pickup_hour"] <= 5
    )
).astype(int)

# Location Pair Feature

In [9]:
fare_df["Location_Pair"] = (
    fare_df["PULocationID"]
    .astype(str)
    +
    "_"
    +
    fare_df["DOLocationID"]
    .astype(str)
)

# 4. Feature Selection

Features selected for modeling are limited to information available before ride initiation.

Trip duration is excluded because it is unavailable at prediction time and would introduce data leakage.

In [10]:
fare_df = fare_df[
    [
        "passenger_count",
        "trip_distance",
        "RatecodeID",
        "cbd_congestion_fee",
        "pickup_hour",
        "pickup_dayofweek",
        "pickup_month",
        "is_weekend",
        "is_rush_hour",
        "is_night",
        "PULocationID",
        "DOLocationID",
        "Location_Pair",
        "fare_amount"
    ]
]

# 5. Data Cleaning

Remove invalid observations and extreme fare values identified during exploratory data analysis.

In [11]:
fare_df = fare_df.dropna()

fare_df = fare_df[
    fare_df["fare_amount"] > 0
]

fare_df = fare_df[
    fare_df["trip_distance"] > 0
]

fare_df = fare_df[
    fare_df["fare_amount"] <= 200
]

print(
    "Final Shape:",
    fare_df.shape
)

Final Shape: (2505383, 14)


# 6. Train Validation Test Split

The dataset is divided into:

- Training Set (80%)
- Validation Set (10%)
- Test Set (10%)

The validation set is used for model comparison, while the test set is reserved for final evaluation.

In [12]:
X = fare_df.drop(
    columns=["fare_amount"]
)

y = fare_df["fare_amount"]

In [13]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [14]:
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

print(X_train.shape)
print(X_valid.shape)
print(X_test.shape)

(2004306, 13)
(250538, 13)
(250539, 13)


# 7. Define Feature Types

In [15]:
numeric_features = [
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "cbd_congestion_fee",
    "pickup_hour",
    "pickup_dayofweek",
    "pickup_month",
    "is_weekend",
    "is_rush_hour",
    "is_night",
]

categorical_features = [
    "PULocationID",
    "DOLocationID",
    "Location_Pair"
]

# 8. Preprocessing Pipelines

Separate preprocessing pipelines are developed for linear and tree-based models.

- Linear Regression requires feature scaling.
- Tree-based models operate effectively without scaling.

In [16]:
linear_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=50
            ),
            categorical_features
        )
    ]
)

In [17]:
tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=50
            ),
            categorical_features
        )
    ]
)

# 9. Evaluation Metrics

Regression performance is assessed using:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R² Score

In [18]:
def regression_metrics(
    y_true,
    y_pred
):

    return {
        "MAE":
        mean_absolute_error(
            y_true,
            y_pred
        ),

        "RMSE":
        np.sqrt(
            mean_squared_error(
                y_true,
                y_pred
            )
        ),

        "R2":
        r2_score(
            y_true,
            y_pred
        )
    }

# 10. Model Development

Three regression algorithms are trained and compared:

- Linear Regression
- XGBoost Regressor
- LightGBM Regressor

In [19]:
models = {
    "Linear Regression": Pipeline(
        steps=[
            (
                "preprocessor",
                linear_preprocessor
            ),
            (
                "model",
                LinearRegression()
            )
        ]
    )
}

In [20]:
models["XGBoost"] = Pipeline(
    steps=[
        (
            "preprocessor",
            tree_preprocessor
        ),
        (
            "model",
            XGBRegressor(
                n_estimators=500,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.85,
                colsample_bytree=0.85,
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [21]:
models["LightGBM"] = Pipeline(
    steps=[
        (
            "preprocessor",
            tree_preprocessor
        ),
        (
            "model",
            LGBMRegressor(
                n_estimators=500,
                learning_rate=0.05,
                subsample=0.85,
                colsample_bytree=0.85,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

# 11. Model Comparison

Validation performance is used to identify the most suitable model for fare prediction.

In [22]:
results = []

trained_models = {}

In [23]:
for model_name, pipeline in models.items():

    print(f"Training {model_name}...")

    pipeline.fit(
        X_train,
        y_train
    )

    y_valid_pred = pipeline.predict(
        X_valid
    )

    metrics = regression_metrics(
        y_valid,
        y_valid_pred
    )

    metrics["Model"] = model_name

    results.append(metrics)

    trained_models[
        model_name
    ] = pipeline

Training Linear Regression...
Training XGBoost...
Training LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.377650 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7826
[LightGBM] [Info] Number of data points in the train set: 2004306, number of used features: 3768
[LightGBM] [Info] Start training from score 19.373542


# 12. Model Comparison

Validation performance is summarized using MAE, RMSE, and R² Score.

The model exhibiting the lowest prediction error and highest explanatory power will be selected for final evaluation.

In [24]:
results_df = (
    pd.DataFrame(results)
    .sort_values(
        by="RMSE"
    )
)

results_df

,MAE,RMSE,R2,Model
2,2.011825,3.647193,0.956350,LightGBM
1,2.068036,3.749361,0.953871,XGBoost
0,2.807854,5.516418,0.900143,Linear Regression


### Interpretation

- Lower MAE indicates smaller average prediction errors.
- Lower RMSE suggests fewer large prediction deviations.
- Higher R² values indicate better explanatory capability.

The best-performing model will be selected based primarily on RMSE and MAE.

# 13. Best Model Selection

The model with the best validation performance is selected for final testing and deployment.

In [25]:
best_model_name = (
    results_df
    .iloc[0]["Model"]
)

best_model = trained_models[
    best_model_name
]

print(
    "Best Model:",
    best_model_name
)

Best Model: LightGBM


# 14. Final Test Set Evaluation

The selected model is evaluated on the unseen test dataset to estimate real-world performance.

In [26]:
test_predictions = best_model.predict(
    X_test
)

In [27]:
test_metrics = regression_metrics(
    y_test,
    test_predictions
)

pd.DataFrame(
    [test_metrics],
    index=[best_model_name]
)

,MAE,RMSE,R2
LightGBM,2.002389,3.581143,0.95777


### Interpretation

Test set performance provides an unbiased estimate of how well the model is expected to generalize to new, unseen trips.

# 15. Overfitting and Generalization Assessment

The selected model is evaluated on both training and test datasets to assess its ability to generalize to unseen data.

Comparing performance across datasets helps identify:

- Overfitting
- Underfitting
- Generalization capability

In [28]:
train_predictions = best_model.predict(
    X_train
)

train_metrics = regression_metrics(
    y_train,
    train_predictions
)

train_metrics

{'MAE': 1.9812771505589541,
 'RMSE': np.float64(3.528120800877823),
 'R2': 0.9590070008637418}

In [29]:
test_predictions = best_model.predict(
    X_test
)

test_metrics = regression_metrics(
    y_test,
    test_predictions
)

test_metrics

{'MAE': 2.0023891597901473,
 'RMSE': np.float64(3.5811431716575366),
 'R2': 0.9577695426280541}

In [30]:
comparison_df = pd.DataFrame(
    [
        train_metrics,
        test_metrics
    ],
    index=[
        "Train",
        "Test"
    ]
)

comparison_df

,MAE,RMSE,R2
Train,1.981277,3.528121,0.959007
Test,2.002389,3.581143,0.957770


In [31]:
r2_gap = (
    train_metrics["R2"]
    - test_metrics["R2"]
)

rmse_gap = (
    test_metrics["RMSE"]
    - train_metrics["RMSE"]
)

print(
    f"R² Gap: {r2_gap:.4f}"
)

print(
    f"RMSE Gap: {rmse_gap:.4f}"
)

R² Gap: 0.0012
RMSE Gap: 0.0530


In [32]:
if (
    train_metrics["R2"] > 0.95
    and r2_gap > 0.10
):
    print(
        "⚠️ Potential Overfitting Detected"
    )

elif (
    train_metrics["R2"] < 0.70
    and test_metrics["R2"] < 0.70
):
    print(
        "⚠️ Potential Underfitting Detected"
    )

else:
    print(
        "✅ Model Generalizes Well"
    )

✅ Model Generalizes Well


### Interpretation

Model performance was compared across training and test datasets to assess generalization capability.

A small difference between training and test performance indicates that the model generalizes well to unseen observations.

Substantial performance degradation on the test set would suggest overfitting, whereas poor performance on both datasets would indicate underfitting.

# 15. Save Trained Model

The final trained pipeline is persisted for future inference and deployment.

In [33]:
from pathlib import Path

PROJECT_DIR = Path(
    "/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/USECASE2_Trip_Fare_Forecasting"
)

MODEL_DIR = PROJECT_DIR / "models"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FARE_MODEL_PATH = MODEL_DIR / "fare_prediction_pipeline.pkl"

In [37]:
joblib.dump(best_model_name, "/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/USECASE2_Trip_Fare_Forecasting/models/fare_model.pkl")

['/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/USECASE2_Trip_Fare_Forecasting/models/fare_model.pkl']

In [34]:
joblib.dump(
    {
        "model_name": best_model_name,
        "pipeline": best_model,
        "numerical_features": numeric_features,
        "categorical_features": categorical_features,
        "test_metrics": test_metrics
    },
    FARE_MODEL_PATH
)

print(
    "Model saved to:",
    FARE_MODEL_PATH
)

Model saved to: /home/ed/Desktop/NYC_Taxi_Demand_Forecasting/USECASE2_Trip_Fare_Forecasting/models/fare_prediction_pipeline.pkl


In [35]:
splits = {
    "X_train": X_train,
    "X_valid": X_valid,
    "X_test": X_test,
    "y_train": y_train,
    "y_valid": y_valid,
    "y_test": y_test,
}

for name, data in splits.items():

    if isinstance(data, pd.Series):
        data.to_frame().to_parquet(
            PROJECT_DIR / "data" / f"{name}.parquet",
            index=False
        )

    else:
        data.to_parquet(
            PROJECT_DIR / "data" / f"{name}.parquet",
            index=False
        )
        